# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/heyzara124-hub/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

*(Rebuilding data, features, and the Week-5 model here so this notebook runs standalone. For
the final playbook I train on all visible rows, not just a train split — this is the version
meant to actually run, not the version meant to be evaluated. Week 5 and Week 6 already show
how it performs on held-out clients.)*

In [1]:
import pandas as pd
import numpy as np
import os

if not os.path.exists('data/raw/content_refresh_anonymized.csv'):
    if not os.path.exists('flyrank-ml-internship'):
        !git clone -q https://github.com/heyzara124-hub/flyrank-ml-internship.git
    os.chdir('flyrank-ml-internship')

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
visible = df[(df['impressions_90d'] >= 500) & (df['avg_position'] > 0) & (df['avg_position'] <= 20)].copy()
tier_median_ctr = visible.groupby('position_tier')['ctr'].median()
visible['tier_median_ctr'] = visible['position_tier'].map(tier_median_ctr)
visible['ctr_gap'] = visible['ctr'] - visible['tier_median_ctr']

numeric_features = [
    'search_volume', 'competition', 'cpc', 'word_count', 'char_count',
    'content_age_days', 'days_since_last_update', 'sessions_90d',
    'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impressions_90d',
]
categorical_features = [
    'content_type', 'main_intent', 'competition_level',
    'freshness_tier', 'position_tier',
]

model_df = visible.copy()
for c in numeric_features:
    model_df[c] = model_df[c].fillna(0)
for c in categorical_features:
    model_df[c] = model_df[c].fillna('unknown')
model_df['log_impressions_90d'] = np.log1p(model_df['impressions_90d'])
numeric_features = [c if c != 'impressions_90d' else 'log_impressions_90d' for c in numeric_features]

X = model_df[numeric_features + categorical_features]
y = model_df['ctr_gap']
print(f'{len(X):,} pages ready for scoring.')

12,023 pages ready for scoring.


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

I trained the same Random Forest from Week 5, this time on all visible pages, and used it to
score every page's `ctr_gap`. Then I gave each flagged page a **reason code**, so a reviewer
never sees a bare number without knowing why it's on the list:

- **`stale_content_review`** → the page hasn't been touched in well over the typical gap —
  action: `refresh_and_republish`.
- **`thin_content_for_tier`** → the page is much shorter than others at its position tier —
  action: `expand_content_depth`.
- **`ctr_below_position_norm`** → neither of the above explains it; the click-through rate is
  just under-performing its position — action: `review_meta_title_snippet` (my Week-4 baseline
  rule's original hypothesis, still the fallback default when nothing more specific stands out).

In [2]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor

pre = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)], remainder='passthrough')
final_model = Pipeline([('pre', pre), ('rf', RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1))])
final_model.fit(X, y)
model_df['predicted_ctr_gap'] = final_model.predict(X)

freshness_median_days = model_df['days_since_last_update'].median()
model_df['wc_med_for_tier'] = model_df.groupby('position_tier')['word_count'].transform('median')

def reason_for(row):
    if row['days_since_last_update'] > freshness_median_days * 1.5:
        return 'stale_content_review', 'refresh_and_republish'
    elif row['word_count'] > 0 and row['word_count'] < row['wc_med_for_tier'] * 0.6:
        return 'thin_content_for_tier', 'expand_content_depth'
    else:
        return 'ctr_below_position_norm', 'review_meta_title_snippet'

reasons = model_df.apply(reason_for, axis=1)
model_df['reason_code'] = [r[0] for r in reasons]
model_df['action_label'] = [r[1] for r in reasons]

queue = model_df.sort_values('predicted_ctr_gap').reset_index(drop=True)
top10 = queue.head(10)

print(f'{len(queue):,} pages in the full queue, worst opportunity first.\n')
print('Top 10:')
print(top10[['content_id', 'position_tier', 'predicted_ctr_gap', 'reason_code', 'action_label', 'impressions_90d']].to_string(index=False))
print('\nAction mix in the top 50:')
print(queue.head(50)['action_label'].value_counts())

12,023 pages in the full queue, worst opportunity first.

Top 10:
          content_id position_tier  predicted_ctr_gap             reason_code              action_label  impressions_90d
content_c82bc0c24241        page_1          -0.210682 ctr_below_position_norm review_meta_title_snippet            13676
content_c6999f7eb5fb        page_1          -0.210237 ctr_below_position_norm review_meta_title_snippet            17790
content_ca17a024f90c        page_1          -0.209025 ctr_below_position_norm review_meta_title_snippet            38815
content_f986bd514b6e        page_1          -0.208212 ctr_below_position_norm review_meta_title_snippet            22456
content_72fdb385e810        page_1          -0.207411    stale_content_review     refresh_and_republish            19360
content_f6ae0f36d70d        page_1          -0.207378 ctr_below_position_norm review_meta_title_snippet            44860
content_11a4f985f14d        page_1          -0.206194 ctr_below_position_norm review_me

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Who:** a FlyRank content editor or SEO reviewer doing their weekly content-review pass. They
open the top 20–50 rows, read the reason code, and decide what to check first.

**What it's for:** prioritization, not decision-making. It answers "what should I look at
first," not "this page is definitely broken."

**Where it stops being valid:**
- Trained on one 30k-row snapshot and one 90-day window — not tested across seasons or over
  time yet, so seasonal content could be mis-ranked.
- Only covers pages that clear the volume floor (`impressions_90d ≥ 500`, position 1–20). Low-
  traffic and very-deep pages aren't scored at all; this queue says nothing about them.
- `ctr_gap` is a proxy. A low gap can mean a genuinely fixable snippet/title problem, or it can
  mean the query intent doesn't match the page — the model can't tell those apart, a human has
  to.
- Reason codes are simple threshold rules layered on top of the model score, not something the
  model itself explains — they're a starting hypothesis, not a diagnosis.

In [3]:
# No computation needed -- this section documents scope, not a calculation.
print('Section 2 is a written scope statement; see the markdown cell above.')

Section 2 is a written scope statement; see the markdown cell above.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**Before acting on any flagged page, a human checks:**
- Does the low CTR actually look like a title/snippet problem when you read the page and the
  query it's ranking for, or does it look like the wrong page for that query (intent mismatch)?
- Is this page already scheduled for a redesign, migration, or removal? Don't duplicate work.
- Is the flagged traffic mostly branded queries (people already searching the client's name)?
  Branded CTR behaves differently and this model wasn't built to handle that case separately.

**Never automated, full stop:**
- No auto-publishing content edits from this queue. A person writes and approves every change.
- No client-facing report that states this model "predicts" or "proves" a page will improve —
  it's a prioritization tool, not a guarantee.
- No using this queue to evaluate a writer's or editor's performance. It flags pages, not
  people, and the causes behind a low score are often outside anyone's control (seasonality,
  competitor changes, algorithm shifts).

In [4]:
# No computation needed -- this section documents review rules, not a calculation.
print('Section 3 is a written review policy; see the markdown cell above.')

Section 3 is a written review policy; see the markdown cell above.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

I'd watch for these signals and retrain or re-check when any of them fire:

- **Precision@50 drop.** If a fresh holdout check (same method as Week 6) falls meaningfully
  below the 0.34 I measured here, the model has drifted from current search behavior.
- **Feature drift.** If the median `word_count`, `days_since_last_update`, or position-tier mix
  in new data shifts a lot from what this model was trained on, its learned patterns may no
  longer apply.
- **Reason code imbalance.** If one reason code starts dominating the queue (say,
  `stale_content_review` jumps from ~20% to 80% of flagged pages), that's more likely a change
  in the underlying content operations than a change in search behavior, and is worth a manual
  look before trusting the queue as-is.
- **Calendar trigger, as a floor.** Even with no warning signs, re-check quarterly — 90-day
  windows this old are already at the edge of what this snapshot can speak to.

In [5]:
# No computation needed -- this section documents monitoring policy, not a calculation.
print('Section 4 is a written monitoring plan; see the markdown cell above.')

Section 4 is a written monitoring plan; see the markdown cell above.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

Per `work/README.md`, dataset CSVs inside `work/` are gitignored on purpose (CI fails if one
gets committed), so I'm not writing the full 12k-row queue as a file. Instead I'm exporting:
a **metrics JSON** (small, committed, the receipts my report's numbers trace back to) and a
**feature-importance chart** (SVG, safe to commit — no row-level data in it). The top-10 table
above is small enough to paste directly into my report as markdown.

In [6]:
import json
import matplotlib.pyplot as plt

os.makedirs('work/outputs', exist_ok=True)
os.makedirs('work/outputs/figures', exist_ok=True)

# metrics JSON -- the receipts, not the raw data
metrics = {
    'queue_size': int(len(queue)),
    'top50_action_mix': queue.head(50)['action_label'].value_counts().to_dict(),
    'avg_predicted_gap_top50': float(queue.head(50)['predicted_ctr_gap'].mean()),
    'distinct_clients_in_top50': int(queue.head(50)['client_id'].nunique()),
    'model_precision_at_50_holdout': 0.34,   # from Week 5 / Week 6, client-holdout split
    'baseline_precision_at_50_holdout': 0.16,  # from Week 4 / Week 6
}
with open('work/outputs/action_playbook_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print('Wrote work/outputs/action_playbook_metrics.json')

# feature importance chart, reused from Week 5
ohe = final_model.named_steps['pre'].named_transformers_['cat']
cat_names = list(ohe.get_feature_names_out(categorical_features))
remainder_names = [c for c in X.columns if c not in categorical_features]
all_names = cat_names + remainder_names
importances = pd.Series(final_model.named_steps['rf'].feature_importances_, index=all_names).sort_values(ascending=False).head(8)

fig, ax = plt.subplots(figsize=(7, 4))
importances.sort_values().plot(kind='barh', ax=ax, color='#5b4fc9')
ax.set_xlabel('Feature importance')
ax.set_title('What the model leans on most')
plt.tight_layout()
plt.savefig('work/outputs/figures/feature_importance.svg')
plt.close()
print('Wrote work/outputs/figures/feature_importance.svg')

Wrote work/outputs/action_playbook_metrics.json
Wrote work/outputs/figures/feature_importance.svg


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.